In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
import json

In [ ]:
radisu = np.linspace(1, 2.4, 15)


In [ ]:
radisu

In [ ]:
'%2d_%.2f'%angle%r

In [ ]:
angles = np.linspace(0, 90, 46)

In [ ]:
angles[34:]

In [ ]:
h = 5
w = 5
angle = 0
radius = 1
print("angle", angle)
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * radius + np.array([0, 0])

m, marker, n_vx, n_edge = periodic_unit_helper.get_zero_area_dashline(h, w, 0.05, dash_point)

finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation=0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation = 1)


fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)


In [ ]:
visualization.plot_2d_mesh(m, pointList=fusedVtx, width=5, height=5)


In [ ]:
visualization.plot_line_segments(n_vx, n_edge)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)


In [ ]:

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

In [ ]:
viewer.show()

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + [ipu.numVars() - 2], 0
fixedVars, hessianShift = [ipu.numVars() - 2], 1e-15
# fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)

ipu.sheet.pressure = 1

#### Problematic cell

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-6


framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr.success

In [ ]:
plt.hist(utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
# name = 'parallel_tube'
# time_stamp = time.strftime("%Y_%m_%d_%H_%M")
# result_folder = 'output/{}/{}'.format(name, time_stamp)
# if not os.path.exists(result_folder):
#     os.makedirs(result_folder)  

az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, True)

In [ ]:
az_ipu = inflation.InflatableMidSurfacePeriodicUnit(m, fusedVtx, epsilon = 1e-5)
az_ipu.ipu.setVars(ipu.getVars())
az_ipu.ipu.sheet.setUseTensionFieldEnergy(True)
az_ipu.ipu.sheet.setUseHessianProjectedEnergy(False)
az_ipu.ipu.sheet.pressure = ipu.sheet.pressure

In [ ]:
az_ipu.energy()

In [ ]:
np.linalg.norm(az_ipu.gradient())

In [ ]:

from tri_mesh_viewer import TriMeshViewer
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)

In [ ]:
az_viewer.show()

In [ ]:
az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
# curr_vars = az_ipu.getVars()
# curr_vars[-2] = 0.3
# curr_vars[-1] = 0
# az_ipu.setVars(curr_vars)

In [ ]:
# az_viewer.saveColorizedObj("bad_base.obj")

In [ ]:
def az_cb(it):
    if it % framerate == 0:
        az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-15
opts.niter = 400
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=az_cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
ipu.get_kappa()

In [ ]:

bending_stiffness_sample_alpha = inflation.getBendingStiffness(az_ipu, [np.pi / 2 if orientation else 0], az_optimizer, 1e-10, [])


In [ ]:
benchmark.reset()
stiffness_shift = 1e-15
while True:
    try:
        stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [])
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
    stiffness_shift *= 10
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
print(az_ipu.energy())

In [ ]:
az_ipu.setVars(save_vars)

In [ ]:
# save_vars = az_ipu.getVars()

In [ ]:
    curr_vars = az_ipu.getVars()
    curr_vars[-2] = 0.1
    curr_vars[-1] = 0
    az_ipu.setVars(curr_vars)

In [ ]:
for i in range(50):
    kappa = (i - 25) / 25 * 0.05
    curr_vars = az_ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = 0
    az_ipu.setVars(curr_vars)
    az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])
    render = az_viewer.offscreenRenderer(1000, 1000)
    render.render()
    render.save("render_{}_{}.png".format(fix_mesh, '%2d'%i))

In [ ]:
stiffness_values = None
benchmark.reset()
stiffness_shift = 1e-15
while True:
    try:
        stiffness_values = inflation.get_equilibrium_sensitivity(az_ipu, 0 if orientation == 0 else np.pi / 2, az_optimizer, hessianShift = stiffness_shift, fixedVars = [])
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
    stiffness_shift *= 10
benchmark.report()


In [ ]:
modes = np.array([stiffness_values]).reshape(len(stiffness_values), 1)
lambdas = [1]

In [ ]:
import mode_viewer, importlib
mview = mode_viewer.ModeViewer(az_ipu, modes, lambdas, amplitude=0.05)
mview.show()

In [ ]:
# curr_vars = az_ipu.getVars()
# curr_vars[-2] = 0.1
# curr_vars[-1] = np.pi / 2
# az_ipu.setVars(curr_vars)

In [ ]:
print(az_ipu.energy())

In [ ]:
print(az_ipu.energy())

In [ ]:
benchmark.reset()
stiffness_shift = 1e-15
while True:
    try:
        stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [])
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
    stiffness_shift *= 10
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
import periodic_simulation_setup

In [ ]:
# bs_obj = periodic_simulation_setup.bending_stiffness_class(az_ipu, az_ipu.ipu.sheet, az_optimizer, viewer, hessianShift = 1e-10, fixedVars = [])

# bs_obj.setVars(bs_obj.getVars())

# fd_validation.secondDerivativeConvergencePlot(bs_obj, epsilons = np.logspace(-6, 1, 100))

In [ ]:
# curr_vars = az_ipu.getVars()
# curr_vars[-1] = np.pi / 2
# curr_vars[-2] = 0.5
# az_ipu.setVars(curr_vars)

In [ ]:
az_viewer.update()

In [ ]:
H = periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian(), reflect = True)

In [ ]:
H[-2, -2]

In [ ]:
bending_stiffness_sample_alpha = inflation.getBendingStiffness(az_ipu, [np.pi / 2], az_optimizer, 1e-10, [])



In [ ]:
bending_stiffness_sample_alpha